# FedSentinel Demo Notebook
**Paper:** FedSentinel: A Byzantine-Resilient Federated Learning Framework with Cryptographic Gradient Attestation Against Coordinated Model Poisoning Attacks

This notebook walks through:
1. Configuration and setup
2. Dataset loading with Dirichlet non-IID partitioning
3. Model architecture
4. CGAP, CADE, and DT-RoA components
5. Mini federated training run (5 rounds for demo)
6. Evaluation and visualization of results from the paper

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Configuration (Table 4)

In [ ]:
from config import *
print('=== FedSentinel Configuration (Table 4) ===')
print(f'Dataset:          {DATASET}')
print(f'Clients N:        {N_CLIENTS}')
print(f'Rounds T:         {N_ROUNDS}')
print(f'Local Epochs K:   {K_LOCAL_EPOCHS}')
print(f'Global LR η:      {GLOBAL_LR}')
print(f'Trust (α,β,γ):    ({TRUST_ALPHA}, {TRUST_BETA}, {TRUST_GAMMA})')
print(f'Byzantine ρ:      {BYZANTINE_FRACTION}')
print(f'Attack:           {ATTACK_TYPE}')

## 2. Dataset Loading and Dirichlet Non-IID Partitioning (Section 6.1)

In [ ]:
from dataset import get_dataloaders, dirichlet_partition
import torchvision

print('Loading CIFAR-10 (auto-download)...')
client_loaders, val_loader, test_loader, root_ds = get_dataloaders(
    dataset_name='cifar10', n_clients=10, batch_size=64, alpha=0.5)

print(f'\nClient shards: {len(client_loaders)}')
print(f'Val size: {len(val_loader.dataset)}')
print(f'Test size: {len(test_loader.dataset)}')
print(f'Root dataset size: {len(root_ds)}')

In [ ]:
# Visualise Dirichlet partition data distribution
n_demo_clients = 10
demo_sizes = [len(l.dataset) for l in client_loaders]

plt.figure(figsize=(10, 3))
plt.bar(range(n_demo_clients), demo_sizes, color='steelblue')
plt.xlabel('Client ID'); plt.ylabel('Samples')
plt.title(f'Non-IID Data Distribution (Dirichlet α={DIRICHLET_ALPHA})')
plt.tight_layout(); plt.show()

## 3. Model Architecture

In [ ]:
from model import build_model, count_parameters

model = build_model('resnet18', num_classes=10)
n_params = count_parameters(model)
print(f'ResNet-18 (CIFAR-10): {n_params:,} parameters')
print(f'Model on: {next(model.parameters()).device}')

# Test forward pass
x = torch.randn(4, 3, 32, 32).to(DEVICE)
with torch.no_grad():
    out = model(x)
print(f'Output shape: {out.shape}')  # (4, 10)

## 4. FedSentinel Components

In [ ]:
from fedsentinel import CGAPModule, CADEModule, DTRoAModule

# CGAP: Cryptographic Gradient Attestation Protocol
cgap = CGAPModule(norm_bound=10.0, delta=0.5, chunk_size=256, bits=8)

# Simulate gradient from a client
d = 100_000  # reduced for demo
fake_grad = torch.randn(d) * 0.05

v_i, grad_q, commitment = cgap.attest(fake_grad.to(DEVICE))
print(f'CGAP attestation result: v_i = {v_i}')
print(f'Quantized gradient shape: {grad_q.shape}')
print(f'Commitment (SHA-256): {commitment.hex()[:32]}...')

In [ ]:
# CADE: Coordinated Attack Detection Engine
cade = CADEModule(proj_dim=64, warmup_rounds=5, spectral_pctl=95, proj_threshold=2.5)

# Simulate honest client gradients
n_clients_demo = 10
honest_grads = torch.randn(n_clients_demo, d) * 0.05

# Inject coordinated attack: 3 clients with correlated malicious gradients
attack_vec = torch.ones(d) / d**0.5
for i in [0, 1, 2]:
    honest_grads[i] = attack_vec * 0.5 + torch.randn(d) * 0.01

score, _ = cade.compute_spectral_score(honest_grads.to(DEVICE))
print(f'CADE spectral score: {score:.4f} (higher = more coordinated)')
print('(Score will trigger detection after warmup calibration)')

In [ ]:
# DT-RoA: Trust scores
dtroa = DTRoAModule(n_clients=n_clients_demo)

v_flags = torch.ones(n_clients_demo, dtype=torch.long)
v_flags[[0, 1, 2]] = 0   # Byzantine fail CGAP

stat_cosines = torch.rand(n_clients_demo)
trust = dtroa.compute_trust(v_flags, stat_cosines, cade_flagged=[])

plt.figure(figsize=(8, 3))
colors = ['tab:red' if i < 3 else 'tab:blue' for i in range(n_clients_demo)]
plt.bar(range(n_clients_demo), trust.numpy(), color=colors)
plt.xlabel('Client ID'); plt.ylabel('Trust Score τ_i')
plt.title('DT-RoA Trust Scores (red = Byzantine, failed CGAP)')
plt.tight_layout(); plt.show()

## 5. Mini Federated Training (5 rounds demo)

In [ ]:
from fedsentinel import FedSentinelServer, FedSentinelClient
from model import get_flat_params
from utils import set_seed, compute_accuracy
import copy

set_seed(42)
global_model = build_model('resnet18', num_classes=10)
server = FedSentinelServer(global_model, n_clients=10, global_lr=0.01)
data_props = torch.ones(10) / 10
root_loader_demo = torch.utils.data.DataLoader(root_ds, batch_size=32, shuffle=True)

demo_rounds = 5
val_accs = []

for t in range(demo_rounds):
    global_params = get_flat_params(global_model).detach().clone()
    client_grads_q = []
    
    for cid in range(10):
        client = FedSentinelClient(cid, client_loaders[cid], global_model)
        _, grad_q, _ = client.local_train(global_params, server.ref_grad)
        client_grads_q.append(grad_q)
    
    agg_grad, info = server.aggregate(client_grads_q, data_props,
                                       server.ref_grad, root_loader_demo)
    server.update_global_model(agg_grad)
    
    val_acc = compute_accuracy(global_model, val_loader)
    val_accs.append(val_acc)
    print(f'Round {t+1:2d} | Val Acc: {val_acc:.2f}% | '
          f'Verified: {info["n_verified"]}/10 | Trust: {info["trust_mean"]:.3f}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, demo_rounds+1), val_accs, marker='o', color='tab:red', linewidth=2)
plt.xlabel('Round'); plt.ylabel('Val Accuracy (%)')
plt.title('FedSentinel Mini Training (5 rounds, 10 clients)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Paper Results Visualisation

In [ ]:
from utils import plot_accuracy_vs_attacks, plot_accuracy_vs_byzantine_fraction
from evaluate import BASELINES

# Table 5 -- Global accuracy under different attacks
methods = list(BASELINES.keys())
attacks = ['GN', 'SF', 'IPM', 'ALIE', 'Min-Max', 'CB', 'CS']
x = np.arange(len(attacks))
width = 0.11

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))
for i, method in enumerate(methods):
    vals = [BASELINES[method].get(a, 0) for a in attacks]
    bar_color = 'tab:red' if method == 'FedSentinel' else colors[i]
    lw = 1.5 if method == 'FedSentinel' else 0
    ax.bar(x + i*width, vals, width, label=method, color=bar_color,
           edgecolor='black', linewidth=lw)

ax.set_xticks(x + width*3.5)
ax.set_xticklabels(attacks, fontsize=11)
ax.set_ylabel('Global Accuracy (%)', fontsize=12)
ax.set_title('Table 5: Global Accuracy on CIFAR-10 (ρ=0.3)', fontsize=13)
ax.legend(fontsize=7.5, ncol=4, loc='lower right')
ax.set_ylim(0, 100)
plt.tight_layout(); plt.show()

In [ ]:
# Table 6 -- Accuracy vs Byzantine fraction
from evaluate import BASELINES

table6 = {
    'FedAvg':     [68.42, 38.27, 14.56, 10.83, 10.12],
    'FLTrust':    [92.18, 88.64, 81.73, 72.35, 63.47],
    'ShieldFL':   [92.56, 89.47, 85.47, 77.62, 69.38],
    'FedSentinel':[93.12, 92.05, 90.38, 86.74, 81.53],
}
fractions = [0.1, 0.2, 0.3, 0.4, 0.45]

plt.figure(figsize=(8, 5))
for method, accs in table6.items():
    lw = 2.5 if method == 'FedSentinel' else 1.5
    ls = '-' if method == 'FedSentinel' else '--'
    c  = 'tab:red' if method == 'FedSentinel' else None
    plt.plot([f*100 for f in fractions], accs, marker='o',
             linewidth=lw, linestyle=ls, color=c, label=method)

plt.xlabel('Byzantine Fraction ρ (%)', fontsize=12)
plt.ylabel('Global Accuracy (%)', fontsize=12)
plt.title('Table 6: Accuracy vs Byzantine Fraction (IPM, CIFAR-10)', fontsize=13)
plt.legend(fontsize=10); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Table 8 -- CADE Detection Performance
from evaluate import CADE_RESULTS
import pandas as pd

df = pd.DataFrame(CADE_RESULTS).T
print('Table 8: CADE Detection Performance (ρ=0.3, CIFAR-10)')
print(df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
attacks_cade = list(CADE_RESULTS.keys())[:-1]
cdrs = [CADE_RESULTS[a]['CDR'] for a in attacks_cade]
fprs = [CADE_RESULTS[a]['FPR'] for a in attacks_cade]

axes[0].bar(attacks_cade, cdrs, color='steelblue')
axes[0].set_title('CADE: Detection Rate (CDR %)')
axes[0].set_ylabel('CDR (%)')
axes[0].set_ylim(80, 100)
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(attacks_cade, fprs, color='coral')
axes[1].set_title('CADE: False Positive Rate (FPR %)')
axes[1].set_ylabel('FPR (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

## Summary

| Component | Role | Key Parameter |
|---|---|---|
| CGAP | Cryptographic norm/direction verification | B=10, δ=0.5 |
| DT-RoA | Trust-weighted aggregation | α=0.3, β=0.4, γ=0.3 |
| CADE | Coalition detection via spectral analysis | k=64, θ_p=2.5 |

**Main result (Table 5, ρ=0.3):**
FedSentinel achieves **91.36% average accuracy**, outperforming the best baseline (ShieldFL: 86.94%) by **4.42%** across 7 attack strategies.

To run full training: `python train.py`  
To evaluate: `python evaluate.py`